In [1]:
import pandas as pd 
import numpy as np
import os 
from pathlib import Path
import json

In [2]:
path = Path("India_runs_data_and_ai_challenge")

In [3]:
with open(path/"candidate_schema.json", "r", encoding = "utf-8") as f:
    schema = json.load(f)

In [4]:
schema.keys()

dict_keys(['$schema', 'title', 'description', 'type', 'required', 'properties'])

In [5]:
candidates = []

with open(path/"candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():  
            candidates.append(json.loads(line))

In [6]:
df = pd.DataFrame(candidates)

In [7]:
df.head()

,candidate_id,profile,career_history,education,skills,certifications,languages,redrob_signals
0,CAND_0000001,"{'anonymized_name': 'Ira Vora', 'headline': 'B...","[{'company': 'Mindtree', 'title': 'Backend Eng...",[{'institution': 'Lovely Professional Universi...,"[{'name': 'Tailwind', 'proficiency': 'intermed...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 86.9, 'signup_d..."
1,CAND_0000002,"{'anonymized_name': 'Saanvi Sethi', 'headline'...","[{'company': 'Wipro', 'title': 'Operations Man...","[{'institution': 'Local Engineering College', ...","[{'name': 'Project Management', 'proficiency':...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 78.7, 'signup_d..."
2,CAND_0000003,"{'anonymized_name': 'Yash Agarwal', 'headline'...","[{'company': 'TCS', 'title': 'Customer Support...","[{'institution': 'Local Engineering College', ...","[{'name': 'Angular', 'proficiency': 'intermedi...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 31.9, 'signup_d..."
3,CAND_0000004,"{'anonymized_name': 'Anil Bose', 'headline': '...","[{'company': 'Dunder Mifflin', 'title': 'Marke...","[{'institution': 'Local Engineering College', ...","[{'name': 'Node.js', 'proficiency': 'intermedi...","[{'name': 'AWS Certified Cloud Practitioner', ...","[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 28.5, 'signup_d..."
4,CAND_0000005,"{'anonymized_name': 'Aisha Sethi', 'headline':...","[{'company': 'Stark Industries', 'title': 'Acc...","[{'institution': 'Chandigarh University', 'deg...","[{'name': 'SQL', 'proficiency': 'beginner', 'e...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 84.6, 'signup_d..."


In [8]:
# Expands the signals dictionary into dedicated columns
df_signals = pd.json_normalize(df['redrob_signals'])
df_signals

,profile_completeness_score,signup_date,last_active_date,open_to_work_flag,profile_views_received_30d,applications_submitted_30d,recruiter_response_rate,avg_response_time_hours,connection_count,endorsements_received,...,skill_assessment_scores.Python,skill_assessment_scores.OpenSearch,skill_assessment_scores.BM25,skill_assessment_scores.scikit-learn,skill_assessment_scores.pgvector,skill_assessment_scores.Embeddings,skill_assessment_scores.LlamaIndex,skill_assessment_scores.PyTorch,skill_assessment_scores.Deep Learning,skill_assessment_scores.QLoRA
0,86.9,2025-10-16,2026-05-20,True,23,2,0.34,177.8,356,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,78.7,2025-07-28,2025-11-12,True,7,1,0.29,171.6,179,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,31.9,2024-08-02,2026-03-21,False,1,9,0.46,119.4,19,46,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28.5,2025-07-21,2026-03-25,False,3,9,0.26,104.1,485,22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,84.6,2023-10-07,2025-10-01,True,12,2,0.37,116.7,300,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,41.4,2022-08-11,2025-12-04,False,52,5,0.45,259.8,325,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99996,42.7,2025-12-09,2026-03-08,True,42,4,0.41,108.6,292,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99997,75.9,2025-10-21,2026-03-07,False,84,0,0.33,139.0,442,66,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99998,49.1,2023-10-14,2025-12-23,True,81,0,0.46,13.7,345,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
import pandas as pd

# 1. Flatten the standard dictionary columns
df_profile = pd.json_normalize(df['profile']).add_prefix('profile_')
df_signals = pd.json_normalize(df['redrob_signals']).add_prefix('signals_')

# 2. Convert list-of-dictionaries into flat, readable comma-separated strings
df['flat_skills'] = df['skills'].apply(lambda x: ", ".join([i.get('name', '') for i in x]) if isinstance(x, list) else "")
df['flat_companies'] = df['career_history'].apply(lambda x: ", ".join([i.get('company', '') for i in x]) if isinstance(x, list) else "")
df['flat_education'] = df['education'].apply(lambda x: ", ".join([i.get('institution', '') for i in x]) if isinstance(x, list) else "")

# 3. Combine everything into one single, non-nested DataFrame
df_flat = pd.concat([
    df['candidate_id'], 
    df_profile, 
    df_signals, 
    df['flat_skills'], 
    df['flat_companies'], 
    df['flat_education']
], axis=1)

# View your newly flattened table
df_flat.head()

,candidate_id,profile_anonymized_name,profile_headline,profile_summary,profile_location,profile_country,profile_years_of_experience,profile_current_title,profile_current_company,profile_current_company_size,...,signals_skill_assessment_scores.scikit-learn,signals_skill_assessment_scores.pgvector,signals_skill_assessment_scores.Embeddings,signals_skill_assessment_scores.LlamaIndex,signals_skill_assessment_scores.PyTorch,signals_skill_assessment_scores.Deep Learning,signals_skill_assessment_scores.QLoRA,flat_skills,flat_companies,flat_education
0,CAND_0000001,Ira Vora,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,Toronto,Canada,6.9,Backend Engineer,Mindtree,10001+,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Tailwind, NLP, Image Classification, Fine-tuni...","Mindtree, Dunder Mifflin",Lovely Professional University
1,CAND_0000002,Saanvi Sethi,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,"Chennai, Tamil Nadu",India,12.5,Operations Manager,Wipro,10001+,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Project Management, React, Photoshop, TypeScri...","Wipro, Wipro, Acme Corp, Dunder Mifflin",Local Engineering College
2,CAND_0000003,Yash Agarwal,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,Austin,USA,1.1,Customer Support,TCS,10001+,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Angular, SEO, Excel, Accounting, Kubernetes, D...",TCS,"Local Engineering College, Chandigarh University"
3,CAND_0000004,Anil Bose,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,Sydney,Australia,3.8,Marketing Manager,Dunder Mifflin,201-500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Node.js, Content Writing, Redux, Airflow, Grap...","Dunder Mifflin, Infosys, Globex Inc","Local Engineering College, Lovely Professional..."
4,CAND_0000005,Aisha Sethi,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,"Gurgaon, Haryana",India,11.0,Accountant,Stark Industries,1001-5000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"SQL, PowerPoint, Photoshop, Tailwind, Apache F...","Stark Industries, Wipro, Initech, TCS",Chandigarh University


In [10]:
def calculate_early_score(row):
    # Extract nested fields safely
    profile = row.get('profile', {}) if isinstance(row.get('profile'), dict) else {}
    history = row.get('career_history', []) if isinstance(row.get('career_history'), list) else []
    skills_list = row.get('skills', []) if isinstance(row.get('skills'), list) else []
    signals = row.get('redrob_signals', {}) if isinstance(row.get('redrob_signals'), dict) else {}
    
    # ----------------------------------------------------
    # HARD DISQUALIFIERS: Professional Behavior Red Flags
    # ----------------------------------------------------
    # 1. Extreme Notice Period
    if signals.get('notice_period_days', 0) > 90:
        return 0
        
    # 2. Low Interview Attendance 
    if signals.get('interview_completion_rate', 1.0) < 0.50:
        return 0
        
    # 3. Complete Unresponsiveness
    if signals.get('recruiter_response_rate', 1.0) < 0.10:
        return 0
    
    # ----------------------------------------------------
    # STAGE 1 SCORING ACCUMULATION (Max 100 points)
    # ----------------------------------------------------
    score = 0
    
    # 1. Location Match (Max 20 pts)
    target_cities = {'pune', 'noida', 'mumbai', 'hyderabad', 'delhi', 'ncr'}
    candidate_loc = str(profile.get('location', '')).lower()
    if any(city in candidate_loc for city in target_cities):
        score += 20
        
    # 2. Platform Availability & Signals (Max 20 pts)
    response_rate = signals.get('recruiter_response_rate', 0)
    score += (response_rate * 15) 
    if signals.get('verified_email', False) or signals.get('verified_phone', False):
        score += 5
        
    # 3. Consulting Firm Filter (Max 20 pts)
    consulting_blacklist = {'tcs', 'tata consultancy', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini'}
    companies = [str(job.get('company', '')).lower() for job in history if job.get('company')]
    if companies:
        all_consulting = all(any(black in comp for black in consulting_blacklist) for comp in companies)
        if not all_consulting:
            score += 20
    else:
        score += 10
        
    # 4. Domain Relevance vs CV/Robotics Trap (Max 20 pts)
    skills_text = " ".join([str(s.get('name', '')).lower() for s in skills_list])
    cv_keywords = {'opencv', 'cnn', 'computer vision', 'ros', 'robotics', 'object detection'}
    ir_keywords = {'embedding', 'retrieval', 'ranking', 'vector', 'search', 'nlp', 'llm', 'ndcg', 'mrr'}
    
    has_cv = any(kw in skills_text for kw in cv_keywords)
    has_ir = any(kw in skills_text for kw in ir_keywords)
    
    if has_cv and not has_ir:
        return 0  # Hard drop pure CV/Robotics profiles
    elif has_ir:
        score += 20
    else:
        score += 10
        
    # 5. Toy RAG / Longevity Check (Max 20 pts)
    has_framework_only = ('langchain' in skills_text or 'openai' in skills_text) and not has_ir
    if not has_framework_only:
        score += 20

    return score

# Re-run the pipeline with behavioral drops included
df['early_score'] = df.apply(calculate_early_score, axis=1)
df_top_1000 = df.nlargest(1000, 'early_score').copy()

print(f"Filtered down to {len(df_top_1000)} candidates after behavior enforcement.")

Filtered down to 1000 candidates after behavior enforcement.


In [11]:
df_top_1000

,candidate_id,profile,career_history,education,skills,certifications,languages,redrob_signals,flat_skills,flat_companies,flat_education,early_score
46131,CAND_0046132,"{'anonymized_name': 'Ishaan Pillai', 'headline...","[{'company': 'Verloop.io', 'title': 'AI Resear...","[{'institution': 'BITS Pilani', 'degree': 'M.E...","[{'name': 'Kubeflow', 'proficiency': 'intermed...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 95.2, 'signup_d...","Kubeflow, Data Science, OpenCV, Information Re...","Verloop.io, Meesho",BITS Pilani,99.10
70524,CAND_0070525,"{'anonymized_name': 'Ved Kumar', 'headline': '...","[{'company': 'Mad Street Den', 'title': 'Senio...","[{'institution': 'VIT Vellore', 'degree': 'M.S...","[{'name': 'Apache Beam', 'proficiency': 'begin...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 95.6, 'signup_d...","Apache Beam, Recommendation Systems, Feature E...","Mad Street Den, Wysa",VIT Vellore,98.95
43636,CAND_0043637,"{'anonymized_name': 'Ela Menon', 'headline': '...","[{'company': 'Rephrase.ai', 'title': 'Junior M...","[{'institution': 'Chandigarh University', 'deg...","[{'name': 'PyTorch', 'proficiency': 'advanced'...","[{'name': 'Deep Learning Specialization', 'iss...","[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 97.5, 'signup_d...","PyTorch, Feature Engineering, Milvus, Redis, H...","Rephrase.ai, Yellow.ai, Genpact AI","Chandigarh University, PES University",98.80
93762,CAND_0093763,"{'anonymized_name': 'Krishna Nair', 'headline'...","[{'company': 'Swiggy', 'title': 'Senior Softwa...","[{'institution': 'NIT Warangal', 'degree': 'B....","[{'name': 'FAISS', 'proficiency': 'advanced', ...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 90.0, 'signup_d...","FAISS, Time Series, LlamaIndex, Vector Search,...","Swiggy, BYJU'S, Swiggy",NIT Warangal,98.80
30,CAND_0000031,"{'anonymized_name': 'Ela Singh', 'headline': '...","[{'company': 'Swiggy', 'title': 'Recommendatio...","[{'institution': 'SRM University', 'degree': '...","[{'name': 'Go', 'proficiency': 'intermediate',...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 83.4, 'signup_d...","Go, MLflow, FAISS, Pinecone, Angular, Image Cl...","Swiggy, Mad Street Den, Uber, Zomato",SRM University,98.65
...,...,...,...,...,...,...,...,...,...,...,...,...
70008,CAND_0070009,"{'anonymized_name': 'Kavya Chopra', 'headline'...","[{'company': 'CRED', 'title': 'Cloud Engineer'...","[{'institution': 'Christ University', 'degree'...","[{'name': 'GraphQL', 'proficiency': 'beginner'...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 53.9, 'signup_d...","GraphQL, Angular, Fine-tuning LLMs, Airflow, K...","CRED, HCL, Wipro, Infosys","Christ University, Local Engineering College",91.00
71457,CAND_0071458,"{'anonymized_name': 'Advik Bansal', 'headline'...","[{'company': 'Wayne Enterprises', 'title': 'Gr...","[{'institution': 'Local Engineering College', ...","[{'name': 'dbt', 'proficiency': 'beginner', 'e...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 50.7, 'signup_d...","dbt, Redis, Docker, Feature Engineering, Agile...","Wayne Enterprises, TCS, Dunder Mifflin, TCS, G...",Local Engineering College,91.00
73026,CAND_0073027,"{'anonymized_name': 'Dhruv Dutta', 'headline':...","[{'company': 'Dunder Mifflin', 'title': 'Busin...","[{'institution': 'RV College of Engineering', ...","[{'name': 'Vue.js', 'proficiency': 'intermedia...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 68.8, 'signup_d...","Vue.js, GraphQL, Kafka, gRPC, JavaScript, Next...",Dunder Mifflin,RV College of Engineering,91.00
79257,CAND_0079258,"{'anonymized_name': 'Sanjay Nair', 'headline':...","[{'company': 'CRED', 'title': 'Software Engine...",[{'institution': 'Delhi College of Engineering...,"[{'name': 'El

In [12]:
def calculate_nice_to_have_bonuses(row):
    score = 0
    
    # Extract structural metadata safely
    history = row.get('career_history', []) if isinstance(row.get('career_history'), list) else []
    signals = row.get('redrob_signals', {}) if isinstance(row.get('redrob_signals'), dict) else {}
    
    # Combine all historical job descriptions and project text into a single block to scan
    history_text = " ".join([
        str(job.get('title', '')).lower() + " " + str(job.get('description', '')).lower() 
        for job in history
    ])
    
    # 1. LLM Fine-Tuning / LoRA Project Bonus (Max 25 pts)
    # Checks for direct project experience with LoRA, QLoRA, or PEFT [cite: 48]
    lora_keywords = {'lora', 'qlora', 'peft', 'fine-tuning', 'fine tuning', 'llm adaptation'}
    if any(kw in history_text for kw in lora_keywords):
        score += 25
        
    # 2. HR-Tech or Marketplace Domain Bonus (Max 25 pts)
    # Looks for prior exposure to talent intelligence, ATS, or marketplaces [cite: 50]
    marketplace_keywords = {'hr-tech', 'hr tech', 'recruiting tech', 'marketplace', 'talent intelligence', 'ats', 'job board'}
    if any(kw in history_text for kw in marketplace_keywords):
        score += 25
        
    # 3. Distributed Systems & Large-Scale Inference Bonus (Max 25 pts)
    # Checks for systems background or inference optimization workloads [cite: 51]
    distribution_keywords = {'distributed systems', 'large-scale inference', 'inference optimization', 'distributed training', 'ray', 'spark', 'cuda'}
    if any(kw in history_text for kw in distribution_keywords):
        score += 25
        
    # 4. Open-Source Contributions Bonus (Max 25 pts)
    # Cross-references explicit history mentions and verified github telemetry [cite: 16, 52]
    has_os_text = any(kw in history_text for kw in {'open-source', 'open source', 'oss contributor'})
    github_score = signals.get('github_activity_score', -1)
    
    if has_os_text or github_score > 30:  # Active github history or explicit text mention [cite: 16, 52]
        score += 25

    return score

# Compute the precise nice-to-have weights on your high-value Stage 1 pool
df_top_1000['nice_to_have_score'] = df_top_1000.apply(calculate_nice_to_have_bonuses, axis=1)

# Sort candidates to bring the ones with actual project depth to the top 
df_final_ordered = df_top_1000.sort_values(by='nice_to_have_score', ascending=False)

In [13]:
df_top_1000

,candidate_id,profile,career_history,education,skills,certifications,languages,redrob_signals,flat_skills,flat_companies,flat_education,early_score,nice_to_have_score
46131,CAND_0046132,"{'anonymized_name': 'Ishaan Pillai', 'headline...","[{'company': 'Verloop.io', 'title': 'AI Resear...","[{'institution': 'BITS Pilani', 'degree': 'M.E...","[{'name': 'Kubeflow', 'proficiency': 'intermed...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 95.2, 'signup_d...","Kubeflow, Data Science, OpenCV, Information Re...","Verloop.io, Meesho",BITS Pilani,99.10,0
70524,CAND_0070525,"{'anonymized_name': 'Ved Kumar', 'headline': '...","[{'company': 'Mad Street Den', 'title': 'Senio...","[{'institution': 'VIT Vellore', 'degree': 'M.S...","[{'name': 'Apache Beam', 'proficiency': 'begin...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 95.6, 'signup_d...","Apache Beam, Recommendation Systems, Feature E...","Mad Street Den, Wysa",VIT Vellore,98.95,25
43636,CAND_0043637,"{'anonymized_name': 'Ela Menon', 'headline': '...","[{'company': 'Rephrase.ai', 'title': 'Junior M...","[{'institution': 'Chandigarh University', 'deg...","[{'name': 'PyTorch', 'proficiency': 'advanced'...","[{'name': 'Deep Learning Specialization', 'iss...","[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 97.5, 'signup_d...","PyTorch, Feature Engineering, Milvus, Redis, H...","Rephrase.ai, Yellow.ai, Genpact AI","Chandigarh University, PES University",98.80,25
93762,CAND_0093763,"{'anonymized_name': 'Krishna Nair', 'headline'...","[{'company': 'Swiggy', 'title': 'Senior Softwa...","[{'institution': 'NIT Warangal', 'degree': 'B....","[{'name': 'FAISS', 'proficiency': 'advanced', ...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 90.0, 'signup_d...","FAISS, Time Series, LlamaIndex, Vector Search,...","Swiggy, BYJU'S, Swiggy",NIT Warangal,98.80,25
30,CAND_0000031,"{'anonymized_name': 'Ela Singh', 'headline': '...","[{'company': 'Swiggy', 'title': 'Recommendatio...","[{'institution': 'SRM University', 'degree': '...","[{'name': 'Go', 'proficiency': 'intermediate',...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 83.4, 'signup_d...","Go, MLflow, FAISS, Pinecone, Angular, Image Cl...","Swiggy, Mad Street Den, Uber, Zomato",SRM University,98.65,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70008,CAND_0070009,"{'anonymized_name': 'Kavya Chopra', 'headline'...","[{'company': 'CRED', 'title': 'Cloud Engineer'...","[{'institution': 'Christ University', 'degree'...","[{'name': 'GraphQL', 'proficiency': 'beginner'...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 53.9, 'signup_d...","GraphQL, Angular, Fine-tuning LLMs, Airflow, K...","CRED, HCL, Wipro, Infosys","Christ University, Local Engineering College",91.00,50
71457,CAND_0071458,"{'anonymized_name': 'Advik Bansal', 'headline'...","[{'company': 'Wayne Enterprises', 'title': 'Gr...","[{'institution': 'Local Engineering College', ...","[{'name': 'dbt', 'proficiency': 'beginner', 'e...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 50.7, 'signup_d...","dbt, Redis, Docker, Feature Engineering, Agile...","Wayne Enterprises, TCS, Dunder Mifflin, TCS, G...",Local Engineering College,91.00,0
73026,CAND_0073027,"{'anonymized_name': 'Dhruv Dutta', 'headline':...","[{'company': 'Dunder Mifflin', 'title': 'Busin...","[{'institution': 'RV College of Engineering', ...","[{'name': 'Vue.js', 'proficiency': 'intermedia...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 68.8, 'signup_d...","Vue.js, GraphQL, Kafka, gRPC, JavaScript, Next...",Dunder Mifflin,RV College of Engineering,91.00,0
79257,CAND_0079258,"{'anonymized_name': 'Sanjay Nair', 'headline':...","[{'company': 'CRED', 'title': 'Software Engine...",[{'institution': 'De

In [14]:
df_top_1000

,candidate_id,profile,career_history,education,skills,certifications,languages,redrob_signals,flat_skills,flat_companies,flat_education,early_score,nice_to_have_score
46131,CAND_0046132,"{'anonymized_name': 'Ishaan Pillai', 'headline...","[{'company': 'Verloop.io', 'title': 'AI Resear...","[{'institution': 'BITS Pilani', 'degree': 'M.E...","[{'name': 'Kubeflow', 'proficiency': 'intermed...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 95.2, 'signup_d...","Kubeflow, Data Science, OpenCV, Information Re...","Verloop.io, Meesho",BITS Pilani,99.10,0
70524,CAND_0070525,"{'anonymized_name': 'Ved Kumar', 'headline': '...","[{'company': 'Mad Street Den', 'title': 'Senio...","[{'institution': 'VIT Vellore', 'degree': 'M.S...","[{'name': 'Apache Beam', 'proficiency': 'begin...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 95.6, 'signup_d...","Apache Beam, Recommendation Systems, Feature E...","Mad Street Den, Wysa",VIT Vellore,98.95,25
43636,CAND_0043637,"{'anonymized_name': 'Ela Menon', 'headline': '...","[{'company': 'Rephrase.ai', 'title': 'Junior M...","[{'institution': 'Chandigarh University', 'deg...","[{'name': 'PyTorch', 'proficiency': 'advanced'...","[{'name': 'Deep Learning Specialization', 'iss...","[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 97.5, 'signup_d...","PyTorch, Feature Engineering, Milvus, Redis, H...","Rephrase.ai, Yellow.ai, Genpact AI","Chandigarh University, PES University",98.80,25
93762,CAND_0093763,"{'anonymized_name': 'Krishna Nair', 'headline'...","[{'company': 'Swiggy', 'title': 'Senior Softwa...","[{'institution': 'NIT Warangal', 'degree': 'B....","[{'name': 'FAISS', 'proficiency': 'advanced', ...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 90.0, 'signup_d...","FAISS, Time Series, LlamaIndex, Vector Search,...","Swiggy, BYJU'S, Swiggy",NIT Warangal,98.80,25
30,CAND_0000031,"{'anonymized_name': 'Ela Singh', 'headline': '...","[{'company': 'Swiggy', 'title': 'Recommendatio...","[{'institution': 'SRM University', 'degree': '...","[{'name': 'Go', 'proficiency': 'intermediate',...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 83.4, 'signup_d...","Go, MLflow, FAISS, Pinecone, Angular, Image Cl...","Swiggy, Mad Street Den, Uber, Zomato",SRM University,98.65,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70008,CAND_0070009,"{'anonymized_name': 'Kavya Chopra', 'headline'...","[{'company': 'CRED', 'title': 'Cloud Engineer'...","[{'institution': 'Christ University', 'degree'...","[{'name': 'GraphQL', 'proficiency': 'beginner'...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 53.9, 'signup_d...","GraphQL, Angular, Fine-tuning LLMs, Airflow, K...","CRED, HCL, Wipro, Infosys","Christ University, Local Engineering College",91.00,50
71457,CAND_0071458,"{'anonymized_name': 'Advik Bansal', 'headline'...","[{'company': 'Wayne Enterprises', 'title': 'Gr...","[{'institution': 'Local Engineering College', ...","[{'name': 'dbt', 'proficiency': 'beginner', 'e...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 50.7, 'signup_d...","dbt, Redis, Docker, Feature Engineering, Agile...","Wayne Enterprises, TCS, Dunder Mifflin, TCS, G...",Local Engineering College,91.00,0
73026,CAND_0073027,"{'anonymized_name': 'Dhruv Dutta', 'headline':...","[{'company': 'Dunder Mifflin', 'title': 'Busin...","[{'institution': 'RV College of Engineering', ...","[{'name': 'Vue.js', 'proficiency': 'intermedia...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 68.8, 'signup_d...","Vue.js, GraphQL, Kafka, gRPC, JavaScript, Next...",Dunder Mifflin,RV College of Engineering,91.00,0
79257,CAND_0079258,"{'anonymized_name': 'Sanjay Nair', 'headline':...","[{'company': 'CRED', 'title': 'Software Engine...",[{'institution': 'De

In [21]:
import pandas as pd
import numpy as np

def calculate_optimized_master_score(row):
    # ----------------------------------------------------
    # 0. SAFELY EXTRACT METADATA
    # ----------------------------------------------------
    profile = row.get('profile', {}) if isinstance(row.get('profile'), dict) else {}
    history = row.get('career_history', []) if isinstance(row.get('career_history'), list) else []
    skills_list = row.get('skills', []) if isinstance(row.get('skills'), list) else []
    signals = row.get('redrob_signals', {}) if isinstance(row.get('redrob_signals'), dict) else {}
    
    # Normalize nested structures into string text blocks for robust phrase scanning
    skills_text = " ".join([str(s.get('name', '')).lower() for s in skills_list])
    history_text = " ".join([
        str(job.get('title', '')).lower() + " " + str(job.get('description', '')).lower() 
        for job in history
    ])
    full_text = skills_text + " " + history_text

    # ----------------------------------------------------
    # 1. STAGE 1: HARD STRATEGIC GATEWAYS & AVAILABILITY (early_score - Max 100)
    # ----------------------------------------------------
    # A. Logistical / Availability Red Flags [cite: 178, 179, 202]
    if signals.get('notice_period_days', 0) > 90:
        return 0
    if signals.get('interview_completion_rate', 1.0) < 0.50:
        return 0
    if signals.get('recruiter_response_rate', 1.0) < 0.10:
        return 0

    # B. THE RELOCATION BLUNDER FIX [cite: 112, 175, 177, 202]
    candidate_loc = str(profile.get('location', '')).lower()
    target_cities = {'pune', 'noida', 'mumbai', 'hyderabad', 'delhi', 'ncr', 'gurgaon'}
    is_in_target_area = any(city in candidate_loc for city in target_cities)
    willing_to_relocate = signals.get('willing_to_relocate', True)

    if not is_in_target_area and not willing_to_relocate:
        return 0  # Hard drop: cannot logistically work a hybrid cadence here

    # C. Pure Consulting Exclusion [cite: 168, 169]
    consulting_blacklist = {'tcs', 'tata consultancy', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini'}
    companies = [str(job.get('company', '')).lower() for job in history if job.get('company')]
    if companies and all(any(black in comp for black in consulting_blacklist) for comp in companies):
        return 0  # Hard drop: entire career spent strictly at outsourcing giants

    # D. Baseline Scoring Elements
    early_score = 0
    if is_in_target_area:
        early_score += 20
    early_score += (signals.get('recruiter_response_rate', 0) * 15)
    if signals.get('verified_email', False) or signals.get('verified_phone', False):
        early_score += 5
    
    # Reward product experience vs mixed history [cite: 169, 188]
    if companies and not any(any(black in comp for black in consulting_blacklist) for comp in companies):
        early_score += 20
    else:
        early_score += 10
    
    # Initial separation of framework wrapper profiles [cite: 142, 166, 197, 198]
    ir_keywords = {'embedding', 'retrieval', 'ranking', 'vector', 'search', 'nlp', 'llm', 'ndcg', 'mrr', 'peft'}
    has_framework_only = ('langchain' in skills_text or 'openai' in skills_text) and not any(kw in skills_text for kw in ir_keywords)
    if not has_framework_only:
        early_score += 40
    else:
        early_score += 10

    # ----------------------------------------------------
    # 2. CORE TECHNICAL MATCH & TENURE STABILITY (core_production_score - Max 100)
    # ----------------------------------------------------
    core_production_score = 0
    
    # Pillar I: Embeddings Operational Competency [cite: 149, 150]
    if any(m in full_text for m in {'sentence-transformers', 'openai embeddings', 'bge', 'e5'}):
        core_production_score += 10
    if any(issue in full_text for issue in {'embedding drift', 'index refresh', 'quality regression', 'retrieval-quality'}):
        core_production_score += 15

    # Pillar II: Vector Database / Hybrid Scale Infrastructure [cite: 151, 152]
    vector_infra = {'pinecone', 'weaviate', 'qdrant', 'milvus', 'opensearch', 'elasticsearch', 'faiss', 'hybrid search'}
    infra_matches = sum(1 for tech in vector_infra if tech in full_text)
    if infra_matches > 0:
        core_production_score += min(10 + (infra_matches * 3), 25)

    # Pillar III: Statistical Ranking Evaluation Frameworks [cite: 134, 154, 155]
    eval_metrics = {'ndcg', 'mrr', 'map', 'offline-to-online', 'correlation', 'a/b test', 'ab testing'}
    eval_matches = sum(1 for metric in eval_metrics if metric in full_text)
    if eval_matches > 0:
        core_production_score += min(10 + (eval_matches * 5), 25)

    # Pillar IV: Average Tenure Length vs. Title Chasing [cite: 164, 165]
    total_years = profile.get('years_of_experience', 0)
    job_count = len(history)
    if job_count > 1:
        avg_tenure = total_years / job_count
        if avg_tenure >= 3.0:
            core_production_score += 25  # Exceptional stability alignment
        elif avg_tenure >= 1.8:
            core_production_score += 15  # Normal startup trajectory standard
    else:
        core_production_score += 20

    # Deflator: Penalize tutorial setups and non-system architects [cite: 144, 166, 167]
    if 'redux' in full_text or ('langchain' in full_text and eval_matches == 0):
        core_production_score = max(0, core_production_score - 20)

    # Balanced CV/Robotics Safeguard [cite: 170, 171]
    cv_gen_keywords = {'opencv', 'cnn', 'computer vision', 'ros', 'robotics', 'object detection', 'diffusion models', 'gans', 'diffusion'}
    core_search_infra = {'vector search', 'hybrid search', 'index refresh', 'embedding drift', 'retrieval-quality', 'weaviate', 'pinecone', 'qdrant', 'milvus', 'faiss', 'ranking', 'ndcg', 'mrr'}
    has_cv_gen = any(kw in full_text for kw in cv_gen_keywords)
    has_core_search = any(kw in full_text for kw in core_search_infra)
    
    if has_cv_gen and not has_core_search:
        core_production_score = max(0, core_production_score - 25)

    # ----------------------------------------------------
    # 3. IDEAL CANDIDATE STRATEGIC TARGETING (ideal_target_score - Max 100)
    # ----------------------------------------------------
    ideal_target_score = 0
    # Sweet spot guide target [cite: 114, 136, 137, 138, 187, 188]
    if 5 <= total_years <= 9:
        ideal_target_score += 20
    if companies and not any(any(black in comp for black in consulting_blacklist) for comp in companies):
        ideal_target_score += 15
        
    system_keywords = {'ranking', 'ranker', 'search', 'recommendation', 'recommender', 'retrieval'}
    if any(kw in history_text for kw in system_keywords):
        ideal_target_score += 25  # Shipped systems matching the core high-level mandate [cite: 127, 128, 189]
        
    fluency_keywords = {'hybrid search', 'dense retrieval', 'offline evaluation', 'online evaluation', 'fine-tuning', 'prompting'}
    fluency_matches = sum(1 for kw in fluency_keywords if kw in full_text)
    ideal_target_score += min(fluency_matches * 5, 20)
    
    # Focus on primary operational hub locations [cite: 112, 175, 191]
    if 'pune' in candidate_loc or 'noida' in candidate_loc:
        ideal_target_score += 10
    if signals.get('open_to_work_flag', False):
        ideal_target_score += 10

    # ----------------------------------------------------
    # 4. NICE-TO-HAVE & COMPLEMENTARY COMPETENCIES (nice_to_have_score - Max 100)
    # ----------------------------------------------------
    nice_to_have_score = 0
    # PEFT & LoRA Adaptation Catch 
    if any(kw in full_text for kw in {'lora', 'qlora', 'peft', 'fine-tuning', 'fine tuning', 'llm adaptation'}):
        nice_to_have_score += 30
    # Domain specific matching context [cite: 159]
    if any(kw in full_text for kw in {'hr-tech', 'hr tech', 'recruiting tech', 'marketplace', 'talent intelligence', 'ats'}):
        nice_to_have_score += 20
    # Large scale framework systems matching [cite: 160]
    if any(kw in full_text for kw in {'distributed systems', 'large-scale inference', 'inference optimization', 'ray', 'spark', 'cuda'}):
        nice_to_have_score += 30
    # Open-Source Validation [cite: 161, 172, 173]
    github_score = signals.get('github_activity_score', -1)
    if 'open-source' in full_text or 'open source' in full_text or github_score > 30:
        nice_to_have_score += 20

    # ----------------------------------------------------
    # 5. PRIORITY WEIGHTED BLEND ARCHITECTURE
    # ----------------------------------------------------
    final_score = (
        (core_production_score * 0.40) +  # 40% Weight: Core Infrastructure Depth & Stability
        (ideal_target_score * 0.30) +     # 30% Weight: Direct Target Profile Alignment
        (nice_to_have_score * 0.20) +     # 20% Weight: Strategic System Bonuses
        (early_score * 0.10)              # 10% Weight: Platform Availability Baselines
    )
    
    return final_score

# 1. Execute single-pass ranking logic across the master pool
df['final_weighted_score'] = df.apply(calculate_optimized_master_score, axis=1)

# 2. Extract the clean, definitive top 100 candidates
df_final_top_100 = df.nlargest(100, 'final_weighted_score').copy()
df_final_top_100 = df_final_top_100.sort_values(by='final_weighted_score', ascending=False)

# 3. Overwrite the file with the clean data
df_final_top_100.to_csv("top_100_founding_candidates.csv", index=False)
print(f"Ecosystem optimization complete. Saved clean recommendations to 'top_100_founding_candidates.csv'.")

Ecosystem optimization complete. Saved clean recommendations to 'top_100_founding_candidates.csv'.


In [22]:
df_top_100

,candidate_id,profile,career_history,education,skills,certifications,languages,redrob_signals,flat_skills,flat_companies,flat_education,early_score,final_weighted_score
39753,CAND_0039754,"{'anonymized_name': 'Mira Banerjee', 'headline...","[{'company': 'Meta', 'title': 'Senior Applied ...","[{'institution': 'IIT Hyderabad', 'degree': 'M...","[{'name': 'Fine-tuning LLMs', 'proficiency': '...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 67.1, 'signup_d...","Fine-tuning LLMs, LlamaIndex, Qdrant, Reinforc...","Meta, Apple, Observe.AI",IIT Hyderabad,77.15,77.215
18498,CAND_0018499,"{'anonymized_name': 'Aarav Trivedi', 'headline...","[{'company': 'Zomato', 'title': 'Senior Machin...",[{'institution': 'Massachusetts Institute of T...,"[{'name': 'Deep Learning', 'proficiency': 'exp...",[{'name': 'Google Cloud Professional ML Engine...,"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 98.0, 'signup_d...","Deep Learning, Weaviate, Recommendation System...","Zomato, Google, Flipkart","Massachusetts Institute of Technology, NIT Sur...",94.15,73.215
81845,CAND_0081846,"{'anonymized_name': 'Arjun Khanna', 'headline'...","[{'company': 'Razorpay', 'title': 'Lead AI Eng...","[{'institution': 'IIT Hyderabad', 'degree': 'B...","[{'name': 'Data Science', 'proficiency': 'adva...",[{'name': 'Google Cloud Professional ML Engine...,"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 95.7, 'signup_d...","Data Science, Information Retrieval, LlamaInde...","Razorpay, Paytm","IIT Hyderabad, IIT Delhi",75.95,73.195
77336,CAND_0077337,"{'anonymized_name': 'Aarav Agarwal', 'headline...","[{'company': 'Paytm', 'title': 'Staff Machine ...","[{'institution': 'Georgia Tech', 'degree': 'B....","[{'name': 'GANs', 'proficiency': 'intermediate...","[{'name': 'Deep Learning Specialization', 'iss...","[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 64.6, 'signup_d...","GANs, Semantic Search, QLoRA, pgvector, Pineco...","Paytm, Razorpay, Glance, Aganitha",Georgia Tech,79.25,70.525
41668,CAND_0041669,"{'anonymized_name': 'Aisha Banerjee', 'headlin...","[{'company': 'CRED', 'title': 'Recommendation ...","[{'institution': 'IIT Guwahati', 'degree': 'B....","[{'name': 'FAISS', 'proficiency': 'advanced', ...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 62.5, 'signup_d...","FAISS, Milvus, PEFT, Diffusion Models, MLflow,...","CRED, Mad Street Den, Yellow.ai","IIT Guwahati, IIT Guwahati",96.55,69.655
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6417,CAND_0006418,"{'anonymized_name': 'Rahul Mukherjee', 'headli...","[{'company': 'Verloop.io', 'title': 'Machine L...","[{'institution': 'Stanford University', 'degre...","[{'name': 'Kubernetes', 'proficiency': 'interm...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 73.5, 'signup_d...","Kubernetes, gRPC, Semantic Search, Embeddings,...","Verloop.io, Flipkart",Stanford University,78.80,49.680
71938,CAND_0071939,"{'anonymized_name': 'Jay Nair', 'headline': 'D...","[{'company': 'Krutrim', 'title': 'Data Scienti...","[{'institution': 'Symbiosis International', 'd...","[{'name': 'TTS', 'proficiency': 'intermediate'...",[],"[{'language': 'English', 'proficiency': 'nativ...","{'profile_completeness_score': 69.8, 'signup_d...","TTS, Elasticsearch, OpenSearch, Embeddings, QL...","Krutrim, Genpact AI",Symbiosis International,95.05,49.605
88387,CAND_0088388,"{'anonymized_name': 'Sai Agarwal', 'headline':...","[{'company': 'Wipro', 'title': 'Senior Data En...","[{'institution': 'SRM University', 'degree': '...","[{'name': 'Image Classification', 'proficiency...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 69.8, 'signup_d...","Image Classification, Reinforcement Learning, ...","Wipro, Nykaa",SRM University,72.05,49.605
79283,CAND_0079284,"{'anonymized_name': 'Ishaan 

In [23]:
# Export the top 100 candidates to a CSV file
df_top_100.to_csv("top_100_founding_candidates.csv", index=False)

print("Successfully exported top 100 candidates to 'top_100_founding_candidates.csv'")

Successfully exported top 100 candidates to 'top_100_founding_candidates.csv'


In [2]:
import pandas as pd
import ast

# 1. Load your optimized top 100 results file
df_top = pd.read_csv("top_100_founding_candidates.csv")

def parse_dict(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return {}
    return val

submission_data = []

# 2. Reformat data to match the sample submission specification
for idx, row in df_top.iterrows():
    prof = parse_dict(row['profile'])
    signals = parse_dict(row['redrob_signals'])
    
    candidate_id = row['candidate_id']
    rank = idx + 1
    
    # Scale score to a 0-1 range decimal for consistency with the sample submission
    score = round(row['final_weighted_score'] / 100.0, 4)
    
    current_title = prof.get('current_title', 'AI Engineer')
    years_exp = prof.get('years_of_experience', 0.0)
    companies = row['flat_companies']
    response_rate = signals.get('recruiter_response_rate', 0.0)
    
    # Extract the top 3 specialized engineering competencies
    skills_list = [s.strip() for s in str(row['flat_skills']).split(',')[:3]]
    top_skills = ", ".join(skills_list)
    
    # Synthesize the exact reasoning string pattern
    reasoning = (
        f"{current_title} with {years_exp:.1f} yrs exp; "
        f"companies: {companies}; "
        f"core infrastructure: {top_skills}; "
        f"response rate: {response_rate:.2f}."
    )
    
    submission_data.append({
        'candidate_id': candidate_id,
        'rank': rank,
        'score': score,
        'reasoning': reasoning
    })

# 3. Create the submission DataFrame and export to disk
df_submission = pd.DataFrame(submission_data)
df_submission.to_csv("final_submission.csv", index=False)

print("Generated final submission matching target format successfully!")

Generated final submission matching target format successfully!
